# Starting cluster

## Prerequisites
First, you need to install MPI, on windows use MS-MPI:
https://msdn.microsoft.com/en-us/library/bb524831(v=vs.85).aspx


## With a profile (not working)
In theory, you should be able to create a profile using
```
ipython profile create --parallel --profile=myprofile
```
and then set
```
c.IPClusterEngines.engine_launcher_class = 'MPIEngineSetLauncher'
```
in ```<IPYTHON-DIR>/profile_myprofile/ipcluster_config.py```. This should then enable you to start a cluster using
```
ipcluster start --profile=myprofile
```
or alternatively through the Clusters tab in Jupyter


## Without a profile (not working)
An alternative is to run
```
ipcluster start --engines=MPI
```


## Manual start (working)
This, however, does *not* work for me on Windows. What does work is the following:

Start a controller using
```
ipcontroller --ip='*'
```
and then start several engines using mpiexec:
```
mpiexec -n 4 ipengine --mpi
```

In [2]:
%setup_logging --out mpi.log my_logger

Console logger using level INFO
File logger using level DEBUG to mpi.log
Python version 3.6.9 |Anaconda, Inc.| (default, Jul 30 2019, 14:00:49) [MSC v.1915 64 bit (AMD64)]


In [3]:
%setup_mpi mpi_context --num_engines 3

Starting IPController
Starting IPEngines
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Waiting for cluster...
Done


In [ ]:
%%px

%matplotlib inline

import numpy as np
from matplotlib import pyplot as plt
from mpi4py import MPI
import json

from GPUSimulators import MPISimulator
from GPUSimulators.common import run_simulation, DataDumper

In [ ]:
%%px
%setup_logging --out "'mpi_' + str(MPI.COMM_WORLD.rank) + '.log'" my_logger

In [ ]:
%%px
%cuda_context_handler my_context

In [ ]:
%%px

def plot_extent(extent, *args, **kwargs):
    x0, x1, y0, y1 = extent
    plt.plot([x0, x1, x1, x0, x0], [y0, y0, y1, y1, y0], *args, **kwargs)

In [ ]:
%%px

import importlib

importlib.reload(MPISimulator)

grid = MPISimulator.MPIGrid(MPI.COMM_WORLD)
print(grid.get_local_rank())

In [1]:
%%px

from GPUSimulators.helpers import InitialConditions

my_context.autotuner = None

nx = 128
ny = 128
gamma = 1.4
save_times = np.linspace(0, 5.0, 10)
outfile = "mpi_out_" + str(MPI.COMM_WORLD.rank) + ".nc"
save_var_names = ['rho', 'rho_u', 'rho_v', 'E']

grid = MPISimulator.MPIGrid(MPI.COMM_WORLD)
print(grid.get_local_rank())

#arguments = InitialConditions.genShockBubble(nx, ny, gamma, grid=grid)
arguments = InitialConditions.genKelvinHelmholtz(nx, ny, gamma, grid=grid)
#arguments = InitialConditions.genRayleighTaylor(nx, ny, gamma, grid=grid)

arguments['context'] = my_context
arguments['theta'] = 1.2
arguments['grid'] = grid

from GPUSimulators import EE2D_KP07_dimsplit


def gen_sim(grid, **kwargs):
    local_sim = EE2D_KP07_dimsplit.EE2D_KP07_dimsplit(**kwargs)
    sim = MPISimulator.MPISimulator(local_sim, grid)
    return sim


outfile = run_simulation(gen_sim, arguments, outfile, save_times, save_var_names)

UsageError: Cell magic `%%px` not found.


In [ ]:
%%px

with DataDumper(outfile, 'r') as indata:
    t = indata.ncfile.variables['time'][:]
    x = indata.ncfile.variables['x'][:]
    y = indata.ncfile.variables['y'][:]

    x_data = grid.gather(x)
    y_data = grid.gather(y)

    created = indata.ncfile.created
    sim_args = json.loads(indata.ncfile.sim_args)
    for key in sim_args:
        if isinstance(sim_args[key], list):
            sim_args[key] = "[...]"
    num_steps = len(t)
    print(f"{outfile} created {created} contains {num_steps} timesteps")
    print("Simulator arguments: \n", sim_args)

    for i in range(num_steps):
        rho = indata.ncfile.variables['rho'][i]
        rho_data = grid.gather(rho)

        #Plot on rank 0
        if grid.comm.rank == 0:
            plt.figure(figsize=(16, 12))
            for k in range(grid.comm.size):
                extent = [x_data[k].min(), x_data[k].max(), y_data[k].min(), y_data[k].max()]
                plt.imshow(rho_data[k], extent=extent, origin='lower', vmin=1.0, vmax=2.0)
                plot_extent(extent, 'k:', alpha=0.1)
            plt.title("t=" + str(t[i]))
            plt.colorbar(shrink=0.33)

In [ ]:
%%px

from GPUSimulators.helpers import InitialConditions
from GPUSimulators.Simulator import BoundaryCondition

nx = 128
ny = 128
g = 9.81
dt = 0.05
width = 50
height = 100
save_times = np.linspace(0, 80, 10)
outfile = "mpi_out_" + str(MPI.COMM_WORLD.rank) + ".nc"
save_var_names = ['h', 'hu', 'hv']

if MPI.COMM_WORLD.rank == 0:
    h0, hu0, hv0, dx, dy = InitialConditions.bump(nx, ny, width, height, g, x_center=0.75, y_center=0.55)
else:
    h0, hu0, hv0, dx, dy = InitialConditions.bump(nx, ny, width, height, g, h_ref=0.5, h_amp=0.0, u_amp=0.0, v_amp=0.0)

bc = BoundaryCondition({
    'north': BoundaryCondition.Type.Reflective,
    'south': BoundaryCondition.Type.Reflective,
    'east': BoundaryCondition.Type.Reflective,
    'west': BoundaryCondition.Type.Reflective
})
grid = MPISimulator.MPIGrid(MPI.COMM_WORLD)

arguments = {
    'context': my_context,
    'h0': h0, 'hu0': hu0, 'hv0': hv0,
    'nx': nx, 'ny': ny,
    'dx': dx, 'dy': dy,
    'g': g,
    'boundary_conditions': bc,
    'grid': grid
}

from GPUSimulators import HLL2


def gen_sim(grid, **kwargs):
    local_sim = HLL2.HLL2(**kwargs)
    sim = MPISimulator.MPISimulator(local_sim, grid)
    return sim


outfile = run_simulation(gen_sim, arguments, outfile, save_times, save_var_names)

In [ ]:
%%px

with DataDumper(outfile, 'r') as indata:
    t = indata.ncfile.variables['time'][:]
    x = indata.ncfile.variables['x'][:]
    y = indata.ncfile.variables['y'][:]

    x_data = grid.gather(x)
    y_data = grid.gather(y)

    created = indata.ncfile.created
    sim_args = json.loads(indata.ncfile.sim_args)
    for key in sim_args:
        if isinstance(sim_args[key], list):
            sim_args[key] = "[...]"
    num_steps = len(t)
    print(f"{outfile} created {created} contains {num_steps} timesteps")
    print("Simulator arguments: \n", sim_args)

    for i in range(num_steps):
        h = indata.ncfile.variables['h'][i]
        #hu = indata.ncfile.variables['hu'][i]
        #hv = indata.ncfile.variables['hv'][i]
        h_data = grid.gather(h)

        #Plot on rank 0
        if grid.comm.rank == 0:
            plt.figure(figsize=(16, 12))
            for k in range(grid.comm.size):
                extent = [x_data[k].min(), x_data[k].max(), y_data[k].min(), y_data[k].max()]
                plt.imshow(h_data[k], extent=extent, origin='lower', vmin=0.49, vmax=0.52)
                plot_extent(extent, 'k:', alpha=0.1)
            plt.title("t=" + str(t[i]))
            plt.colorbar(shrink=0.33)